In [1]:
import pandas as pd
import numpy as np

from scipy.optimize import brentq
from scipy.stats import norm

from typing import Dict, Any

In [ ]:
def build_fixing_schedule(front_future, fixing_month):
    # front_future = [Apr21, Dec22] -> take the prices FROM THE EXCEL (expiry coincides with the last value)
    # front_future: pandas Series with the settlement prices of the front future, indexed by date

    # fixing_month = [Feb 2021, Oct 2022] (training and test set) 
    # fixing_month: string in the format: "YYYY-MM"

    # output: n1, n2, w1, w2

    days = front_future.loc[fixing_month].index      # N days
    expiry = front_future.index.max()                # last date of the future series
    last_front = days[days < expiry][-2]             # second-to-last fixing before expiry (Annex B)
    n1 = int((days <= last_front).sum()); n2 = len(days) - n1
    return {"days": days, "n1": n1, "n2": n2,
            "w1": n1 / len(days), "w2": n2 / len(days)}

In [3]:
# Esempio DA TOGLIERE (più o meno cosa scriveremo nel main)
path = "C:/Users/arian/Desktop/Magistrale/Secondo anno/Computational Finance/Azzone/Hedging competition/COJ21_COZ22_historical_series.xlsx"

# training: front leg = Apr21 (COJ21 sheet), averaging month Feb 2021
fix_apr = pd.read_excel(path, sheet_name="COJ21").set_index("date")["settlement"]
sched_train = build_fixing_schedule(fix_apr, "2021-02")
print(sched_train["n1"])


18


In [ ]:
def black76(F, K, T, sigma, typ):
    # undiscounted Black-76 (r = 0)
    # T is the time to maturity, not the maturity!
    d1 = (np.log(F / K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call = F * norm.cdf(d1) - K * norm.cdf(d2)
    # put from put-call parity with r = 0: P = C - (F - K)
    return call if typ == "Call" else call - (F - K)

def implied_vol(price, F, K, T, typ):
    # find sigma such that the model price equals the market price
    f = lambda s: black76(F, K, T, s, typ) - price
    try:
        return brentq(f, 1e-4, 3.0)      # wide bracket for sigma
    except ValueError:                   # no root in the bracket (bad quote)
        return np.nan

In [5]:
def build_context(
    df: pd.DataFrame,
    front_future: pd.Series,
    config: dict,
    fit_market,        # function: calibrates the model on one day (model-specific)
    price_option,      # function: prices ONE option on the basket (model-specific)
) -> Dict[str, Any]:
    # 1. inputs and model functions
    ctx = {
        "df": df,
        "cfg": config,
        "front_future": front_future,
        "k_call": config["k_call"],
        "fit_market": fit_market,
        "price_option": price_option,
    }

    # 2. quantities that do not change during the backtest, added ONE AT A TIME:
    #    each function receives ctx, which already contains what it needs
    ctx["sched"] = build_fixing_schedule(front_future, config["fixing_month"])
    ctx["rho_series"] = compute_expanding_correlation(ctx)   # needs df, cfg, front_future
    ctx["k_put"] = solve_k_put(ctx)                          # needs sched, rho_series, and calibrate(ctx, date)

    return ctx

In [10]:
# Esempio DA TOGLIERE (più o meno cosa scriveremo nel main)
df = pd.read_parquet("C:/Users/arian/Desktop/Magistrale/Secondo anno/Computational Finance/Azzone/Hedging competition/unpivoted_bid_offer_train_clean.parquet")
config = {
    "contracts": ("COJ21", "COK21"),   # the two legs: front first, next second
    "fixing_month": "2021-02",         # averaging month, format "YYYY-MM"
    "k_call": 78.0,                    # call strike fixed 
    "start": "2020-01-06",             # inception date of the collar
    "end": "2021-02-28",               # last backtest date
}

In [ ]:
def compute_expanding_correlation(ctx):
     # Computes the expanding correlation for ALL dates at once.
     # It is called a single time, in build_context, and the result is stored
     # in ctx["rho_series"]. In this way we avoid to rebuild the returns and the whole 
     # expanding window from scratch at each backtest date

     second_leg = (
             ctx["df"][ctx["df"]["actual_underlying_ticker"] == ctx["cfg"]["contracts"][1]]
             .drop_duplicates("date")
             .set_index("date")["underlying_settlement"]
        )

     both = pd.concat([ctx["front_future"].rename("a"), second_leg.rename("b")], axis=1).dropna()  # joint settlement series of the two legs
     log_ret = np.log(both).diff().dropna()

     return log_ret["a"].expanding(min_periods=20).corr(log_ret["b"]) # correlation on an expanding window, NaN until at least 20 returns are available

def get_rho(ctx, date):
     # Takes the last value of the expanding correlation up to that date (no look-ahead);
     # if there are fewer than 20 returns (all values are NaN) we use the fixed 0.99.
     r = ctx["rho_series"].loc[:date].dropna()
     return r.iloc[-1] if len(r) else 0.99



In [ ]:
def make_smile_logN(strikes, ivs, F):
    # smooth smile: sigma(x) = c2*x^2 + c1*x + c0, with x = ln(K/F), fitted by least squares
    coef = np.polyfit(np.log(strikes / F), ivs, 2)
    return lambda K: np.polyval(coef, np.log(K / F))

def fit_lognormal(df, date, contracts):
    # model-specific: from the quotes of one day to the smile of each leg
    # output: dict with "F1", "F2", "sig1", "sig2" (sig_ as a function of K)
    # qui lo scopo principale è quello di costruire lo smile come funzione di K
    # model specific make_smile
    legs = [] # list to be filled with the couple (F, smile) for every leg -> 2 couples
    for tk in contracts: # contracts are in config (for training: COJ21 and COK21)
        s = df[(df["date"] == date) & (df["actual_underlying_ticker"] == tk)].copy() # set of option prices, at date, with underlying one of the 2 contract
        s = s[(s["option_bid"] > 0) & (s["option_ask"] >= s["option_bid"])]   # liquidity filter
        F = s["underlying_settlement"].iloc[0] # equal on every row -> we consider only the first (same date)
        T = (s["expiration"].iloc[0] - date).days / 365 # time to maturity
        s["mid"] = (s["option_bid"] + s["option_ask"]) / 2
        s = s[((s["option_type"] == "Put") & (s["strike"] < F)) |
              ((s["option_type"] == "Call") & (s["strike"] >= F))]            # OTM only
        s["iv"] = [implied_vol(m, F, K, T, ty) for m, K, ty
                   in zip(s["mid"], s["strike"], s["option_type"])]
        s = s.dropna(subset=["iv"]).sort_values("strike") # np.interp requires increasing strike
        legs.append((F, make_smile_logN(s)))
    (F1, sig1), (F2, sig2) = legs
    return {"F1": F1, "F2": F2, "sig1": sig1, "sig2": sig2}

In [ ]:

def calibrate(ctx, date):
    # generic: model part + what does not depend on the model (rho, date)
    date = pd.Timestamp(date) # convert a string in Timestamp
    state = ctx["fit_market"](ctx["df"], date, ctx["cfg"]["contracts"]) # read the function associated to fit_market
    state["rho"] = get_rho(ctx, date)
    state["date"] = date
    return state


In [ ]:
def price_lognormal(ctx, K, typ, state, F1, F2, t0):
        # price of ONE option (call or put), delegated to the model
        # output: option price per barrel

        sig1, sig2, rho = state["sig1"], state["sig2"], state["rho"]
        n1, n2 = ctx["sched"]["n1"], ctx["sched"]["n2"]
        w1, w2 = ctx["sched"]["w1"], ctx["sched"]["w2"]
        Favg = w1 * F1 + w2 * F2

        # equivalent strike on each leg, then vol read from that leg's smile
        s1, s2 = sig1(F1 * K / Favg), sig2(F2 * K / Favg)

        # year fractions from the valuation date t0 to each fixing date
        th = np.array([(d - t0).days / 365 for d in ctx["sched"]["days"]])
        th1, th2 = th[:n1], th[n1:]
        k = np.arange(1, n1 + 1); j = np.arange(1, n2 + 1)

        # variances and covariance of the two averages (eq. 15, 16, 18 of the Note)
        var1 = F1**2 / n1**2 * np.sum((2*(n1 - k) + 1) * (np.exp(s1**2 * th1) - 1))
        var2 = F2**2 / n2**2 * np.sum((2*(n2 - j) + 1) * (np.exp(s2**2 * th2) - 1))
        cov = F1 * F2 / n1 * np.sum(np.exp(rho * s1 * s2 * th1) - 1)
        varA = w1**2 * var1 + w2**2 * var2 + 2 * w1 * w2 * cov

        # lognormal fit and Black-76 type formula (eq. 21-24), no discounting
        v = np.sqrt(np.log(1 + varA / Favg**2))
        d1 = (np.log(Favg / K) + 0.5 * v**2) / v
        d2 = d1 - v
        call = Favg * norm.cdf(d1) - K * norm.cdf(d2)
        return call if typ == "Call" else call - (Favg - K)   # put via parity


In [ ]:
def collar_value(ctx, state, F1, F2, date):
        # collar value PER BARREL for the market-maker: long put, short call
        # since it is marked to market, P&L includes (nominal) * (value_t - value_t-1)
        # we will use the value of the collar also for the computation of the delta
        # (la delta per il hedging si calcola riprezzando il collar con i future spostati)

        price = ctx["price_option"]
        return (price(ctx, ctx["k_put"], "Put", state, F1, F2, date)
          - price(ctx, ctx["k_call"], "Call", state, F1, F2, date))
    
def solve_k_put(ctx):
        # zero-cost collar: find K_put such that Put(K_put) = Call(k_call) at inception
        t0 = pd.Timestamp(ctx["cfg"]["start"])
        st = calibrate(ctx, t0)
        price = ctx["price_option"]
        c_star = price(ctx, ctx["k_call"], "Call", st, st["F1"], st["F2"], t0)   # call priced once
        h = lambda K: price(ctx, K, "Put", st, st["F1"], st["F2"], t0) - c_star
        favg = ctx["sched"]["w1"] * st["F1"] + ctx["sched"]["w2"] * st["F2"]
        return brentq(h, 30.0, favg)

In [ ]:
# esempio
ctx = build_context(df, fix_apr, config, fit_lognormal, price_lognormal)

In [ ]:
def greeks(ctx, state, date, h=0.5):
        # Sensitivities of the collar value, by central finite differences
        # h : bump size on each future, in dollars
        # dict {"delta1": ..., "delta2": ...}, derivative of value per barrel with
        # respect to F1 and F2 (both negative for the market maker). A "vega" key can
        # be added later if the hedge uses options.
        F1, F2 = state["F1"], state["F2"]
        v = lambda a, b: collar_value(ctx, state, a, b, date)
        return {"delta1": (v(F1 + h, F2) - v(F1 - h, F2)) / (2 * h),
                "delta2": (v(F1, F2 + h) - v(F1, F2 - h)) / (2 * h)}